In [1]:
# Setup: load baseline/bandit CSVs + FLEC JSONL (flec_metrics_v2) into one DataFrame

import os
import sys
import json
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure we can import python/experiments/flec_metrics.py when running from repo root.
EXPERIMENTS_DIR = (Path.cwd() / 'python' / 'experiments').resolve()
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

from flec_metrics import (
    flec_corrected_attempted_bytes,
    flec_corrected_e2e_delay_s,
    flec_corrected_goodput_mbps,
    flec_corrected_overhead_ratio,
)

# -------------------------
# User-editable parameters
# -------------------------
SELECT_METHODS = [
    # BCIR
    'bandit',
    # QUIC
    'quic_bbrv2',
    # IR-FEC1 / IR-FEC2 (edit these to match your experiment method names if needed)
    'fec_k30_r0_2_rstep_6',
    'fec_k30_r0_10_rstep_6',
    # FLEC
    'flec',
]

FLEC_E2E_OFFSET_MS = 0.0  # can be negative; scenario-dependent
os.environ['FLEC_E2E_OFFSET_MS'] = str(float(FLEC_E2E_OFFSET_MS))

TIMEOUT_S = 1.0
TIMEOUT_MS = float(TIMEOUT_S) * 1000.0

DDL_MS_LIST = [200, 300, 400, 500]
GE_DDL_MS = 350

TASK_DELAY = 'delay_128kb'
TASK_GOODPUT_PREFERRED = 'file_1048576B'

# Plot styling
METHOD_LABEL = {
    'bandit': 'BCIR',
    'quic_bbrv2': 'QUIC',
    'fec_k30_r0_2_rstep_6': 'IR-FEC1',
    'fec_k30_r0_10_rstep_6': 'IR-FEC2',
    'flec': 'FLEC',
}

METHOD_ORDER = SELECT_METHODS[:]

def _method_label(m: str) -> str:
    return METHOD_LABEL.get(str(m), str(m))

# -------------------------
# Parsing helpers
# -------------------------
_GE_RE = re.compile(r'^gemodel:(?P<p>[^,]+),(?P<r>[^,]+),(?P<x>[^,]+),(?P<rtt>[^,]+)')

def format_ge_loss_mode(p_pct: float, r_pct: float, rtt_ms: float) -> str:
    return f'gemodel:{float(p_pct):.6f},{float(r_pct):.6f},0.000000,{float(rtt_ms):.6f}'

def parse_ge_loss_mode(loss_mode: str):
    m = _GE_RE.match(str(loss_mode or ''))
    if not m:
        return None
    try:
        return {
            'pi_bad_pct': float(m.group('p')),
            'r_pct': float(m.group('r')),
            'rtt_ms': float(m.group('rtt')),
        }
    except Exception:
        return None

def parse_iid_loss_mode(loss_mode: str):
    s = str(loss_mode or '').strip()
    if not s.startswith('iid:'):
        return None
    try:
        return float(s.split(':', 1)[1])
    except Exception:
        return None

# -------------------------
# Loaders
# -------------------------
def load_baseline_csvs() -> pd.DataFrame:
    rows = []
    for p in sorted(glob.glob('python/results/*-baseline-data/results.csv')):
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        df['source_path'] = p
        df['source_kind'] = 'baseline'
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    # Keep only the columns we need (missing columns become NaN).
    keep = [
        'task','method','sender_id','loss_mode','rep',
        'success','timed_out','md5_ok','client_ok',
        'dur_ms','e2e_delay_ms','goodput_mbps','overhead_ratio',
        'source_kind','source_path',
    ]
    for k in keep:
        if k not in out.columns:
            out[k] = np.nan
    return out[keep]

def load_bandit_csvs() -> pd.DataFrame:
    rows = []
    for p in sorted(glob.glob('python/results/*-bandit-data-*/bandit_eval_results.csv')):
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        df['source_path'] = p
        df['source_kind'] = 'bandit'
        # Normalize to baseline-like schema.
        if 'method' not in df.columns:
            df['method'] = 'bandit'
        df['method'] = 'bandit'
        keep = [
            'task','method','sender_id','loss_mode','rep',
            'success','dur_ms','e2e_delay_ms','goodput_mbps','overhead_ratio',
            'source_kind','source_path',
        ]
        for k in keep:
            if k not in df.columns:
                df[k] = np.nan
        rows.append(df[keep])
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

def load_flec_jsonls() -> pd.DataFrame:
    paths = sorted(glob.glob('python/results/flec_data/*.jsonl'))
    out_rows = []
    for p in paths:
        with open(p, 'r', encoding='utf-8') as f:
            for line in f:
                line = (line or '').strip()
                if not line:
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                if not isinstance(d, dict):
                    continue

                ok = int(d.get('ok', 0) or 0)
                delay_s = flec_corrected_e2e_delay_s(d)
                overhead = flec_corrected_overhead_ratio(d)
                goodput = flec_corrected_goodput_mbps(d)

                # sender may be None in IID logs; match baseline IID sender_id=0.
                sender_raw = d.get('sender', 0)
                try:
                    sender_id = int(sender_raw) if sender_raw is not None else 0
                except Exception:
                    sender_id = 0

                rep_raw = d.get('trial', 0)
                try:
                    rep = int(rep_raw)
                except Exception:
                    rep = 0

                # Determine task based on tx_data_bytes if present.
                data_bytes = d.get('tx_data_bytes', None)
                try:
                    data_i = int(data_bytes) if data_bytes is not None else 0
                except Exception:
                    data_i = 0
                if data_i == 128 * 1024:
                    task = 'delay_128kb'
                elif data_i > 0:
                    task = f'file_{data_i}B'
                else:
                    task = ''

                # Loss mode normalization.
                loss_model = str(d.get('loss_model', '') or '').strip().lower()
                p_pct = d.get('p_pct', None)
                r_pct = d.get('r_pct', None)
                loss_pct = d.get('loss_pct', None)
                rtt_ms = d.get('rtt_ms', None)
                try:
                    rtt_ms_f = float(rtt_ms) if rtt_ms is not None else 0.0
                except Exception:
                    rtt_ms_f = 0.0

                loss_mode = ''
                if loss_model == 'ge' or (p_pct is not None and r_pct is not None):
                    try:
                        loss_mode = format_ge_loss_mode(float(p_pct), float(r_pct), float(rtt_ms_f))
                    except Exception:
                        loss_mode = 'gemodel:'
                else:
                    # IID: percent units (e.g. 0.1, 0.2, ...)
                    try:
                        lp = float(loss_pct) if loss_pct is not None else float(p_pct)
                        loss_mode = f'iid:{lp:g}'
                    except Exception:
                        loss_mode = 'iid:'

                out_rows.append({
                    'task': task,
                    'method': 'flec',
                    'sender_id': sender_id,
                    'loss_mode': loss_mode,
                    'rep': rep,
                    'success': ok,
                    'timed_out': 0 if ok == 1 else 1,
                    'dur_ms': int(round(float(delay_s) * 1000.0)) if delay_s is not None else 0,
                    'e2e_delay_ms': float(delay_s) * 1000.0 if delay_s is not None else np.nan,
                    'goodput_mbps': float(goodput) if goodput is not None else np.nan,
                    'overhead_ratio': float(overhead) if overhead is not None else np.nan,
                    'source_kind': 'flec',
                    'source_path': p,
                })

    if not out_rows:
        return pd.DataFrame()
    return pd.DataFrame(out_rows)

baseline_df = load_baseline_csvs()
bandit_df = load_bandit_csvs()
flec_df = load_flec_jsonls()

df = pd.concat([x for x in [baseline_df, bandit_df, flec_df] if x is not None and len(x) > 0], ignore_index=True)

# Normalize types
df['method'] = df['method'].astype(str)
df['task'] = df['task'].astype(str)
df['loss_mode'] = df['loss_mode'].astype(str)
df['sender_id'] = pd.to_numeric(df['sender_id'], errors='coerce').fillna(0).astype(int)
df['success'] = pd.to_numeric(df['success'], errors='coerce').fillna(0).astype(int)
df['timed_out'] = pd.to_numeric(df.get('timed_out', 0), errors='coerce').fillna(0).astype(int)
df['e2e_delay_ms'] = pd.to_numeric(df['e2e_delay_ms'], errors='coerce')
df['overhead_ratio'] = pd.to_numeric(df['overhead_ratio'], errors='coerce')
df['goodput_mbps'] = pd.to_numeric(df['goodput_mbps'], errors='coerce')

df['scenario'] = np.where(df['loss_mode'].str.startswith('gemodel:'), 'ge', 'iid')
df['method_label'] = df['method'].map(_method_label)

# Derived fields for plotting
df['iid_loss_pct'] = df['loss_mode'].apply(parse_iid_loss_mode)
df['pi_bad_pct'] = df['loss_mode'].apply(lambda s: (parse_ge_loss_mode(s) or {}).get('pi_bad_pct', np.nan))

# Apply method selection
df_sel = df[df['method'].isin(SELECT_METHODS)].copy()

# Auto-pick goodput task
if (df_sel['task'] == TASK_GOODPUT_PREFERRED).any():
    TASK_GOODPUT = TASK_GOODPUT_PREFERRED
else:
    # fall back: any file_* task, else delay task
    file_tasks = sorted({t for t in df_sel['task'].unique() if str(t).startswith('file_')})
    TASK_GOODPUT = file_tasks[-1] if file_tasks else TASK_DELAY

print('Loaded rows:', len(df))
print('Selected rows:', len(df_sel))
print('Selected methods:', SELECT_METHODS)
print('Tasks present:', sorted(df_sel['task'].unique().tolist()))
print('Goodput task:', TASK_GOODPUT)

ValueError: No objects to concatenate

In [ ]:
# 1) E2E delay CDF



CDF_SCENARIO = 'ge'  # 'ge' or 'iid'

INCLUDE_FAILURES = False

DDL_MS_FOR_CDF = 500  # used only when INCLUDE_FAILURES=True (failures get clamped here)



sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == CDF_SCENARIO)].copy()

sub = sub[np.isfinite(sub['e2e_delay_ms'])]



by = {}

for m in METHOD_ORDER:

    dm = sub[sub['method'] == m]

    if INCLUDE_FAILURES:

        ok = dm[(dm['success'] == 1) & (dm['e2e_delay_ms'] > 0)]['e2e_delay_ms'].tolist()

        fail_n = int((dm['success'] != 1).sum())

        xs = ok + [float(DDL_MS_FOR_CDF)] * fail_n

    else:

        xs = dm[(dm['success'] == 1) & (dm['e2e_delay_ms'] > 0)]['e2e_delay_ms'].tolist()

    xs = [x for x in xs if np.isfinite(x) and x > 0]

    by[m] = xs



def ecdf(vals):

    x = np.asarray([v for v in vals if np.isfinite(v)], dtype=float)

    if x.size == 0:

        return np.asarray([]), np.asarray([])

    x = np.sort(x)

    y = np.arange(1, x.size + 1, dtype=float) / float(x.size)

    return x, y



plt.figure(figsize=(5.2, 3.0))

for m in METHOD_ORDER:

    vals = by.get(m, [])

    if not vals:

        continue

    x, y = ecdf(vals)

    plt.plot(x, y, label=_method_label(m))

plt.xlabel('E2E delay (ms)')

plt.ylabel('CDF')

plt.title(f'E2E delay CDF ({CDF_SCENARIO}, task={TASK_DELAY})')

plt.grid(True, alpha=0.25)

plt.legend()

plt.tight_layout()

plt.show()


In [ ]:
# 2) Overhead boxplot

BOX_SCENARIO = 'ge'  # 'ge' or 'iid'
sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == BOX_SCENARIO) & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['overhead_ratio']) & (sub['overhead_ratio'] >= 0)]
sub['method_label'] = pd.Categorical(sub['method'].apply(_method_label), categories=[_method_label(m) for m in METHOD_ORDER], ordered=True)

plt.figure(figsize=(6.2, 3.2))
sns.boxplot(data=sub, x='method_label', y='overhead_ratio')
plt.xlabel('Method')
plt.ylabel('Overhead ratio')
plt.title(f'Overhead boxplot ({BOX_SCENARIO}, task={TASK_DELAY})')
plt.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# 3) Completion ratio vs DDL

CR_SCENARIO = 'ge'  # 'ge' or 'iid'
sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == CR_SCENARIO)].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms'])] 

def completion_ratio(df_in: pd.DataFrame, ddl_ms: float) -> float:
    if len(df_in) == 0:
        return np.nan
    complete = (df_in['success'] == 1) & (df_in['e2e_delay_ms'] > 0) & (df_in['e2e_delay_ms'] <= float(ddl_ms))
    return float(complete.mean())

plt.figure(figsize=(5.8, 3.2))
for m in METHOD_ORDER:
    dm = sub[sub['method'] == m]
    ys = [completion_ratio(dm, ddl) for ddl in DDL_MS_LIST]
    plt.plot(DDL_MS_LIST, ys, marker='o', label=_method_label(m))
plt.ylim(0.0, 1.02)
plt.xlabel('DDL (ms)')
plt.ylabel('Completion ratio')
plt.title(f'Completion ratio vs DDL ({CR_SCENARIO}, task={TASK_DELAY})')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4) Overhead vs E2E delay scatter (per-trial or aggregated)

SCATTER_SCENARIO = 'ge'
SCATTER_AGG = 'sender_loss_method_mean'  # 'per_trial' or 'sender_loss_method_mean'

sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == SCATTER_SCENARIO) & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms']) & (sub['e2e_delay_ms'] > 0)]
sub = sub[np.isfinite(sub['overhead_ratio']) & (sub['overhead_ratio'] >= 0)]

if SCATTER_AGG == 'sender_loss_method_mean':
    grp = sub.groupby(['sender_id', 'loss_mode', 'method'], as_index=False).agg({
        'e2e_delay_ms': 'mean',
        'overhead_ratio': 'mean',
        'goodput_mbps': 'mean',
    })
else:
    grp = sub

grp['method_label'] = grp['method'].apply(_method_label)

plt.figure(figsize=(6.2, 3.4))
sns.scatterplot(data=grp, x='overhead_ratio', y='e2e_delay_ms', hue='method_label')
plt.xlabel('Overhead ratio')
plt.ylabel('E2E delay (ms)')
plt.title(f'Overhead vs E2E delay ({SCATTER_SCENARIO}, {SCATTER_AGG})')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5) Delay percentile bar chart (p50 and p99)

PCT_SCENARIO = 'ge'
sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == PCT_SCENARIO) & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms']) & (sub['e2e_delay_ms'] > 0)]

rows = []
for m in METHOD_ORDER:
    xs = sub[sub['method'] == m]['e2e_delay_ms'].to_numpy(dtype=float)
    xs = xs[np.isfinite(xs)]
    if xs.size == 0:
        continue
    rows.append({
        'method': _method_label(m),
        'p50': float(np.percentile(xs, 50)),
        'p99': float(np.percentile(xs, 99)),
    })
pct_df = pd.DataFrame(rows)

plt.figure(figsize=(6.2, 3.2))
x = np.arange(len(pct_df))
w = 0.35
plt.bar(x - w/2, pct_df['p50'], width=w, label='p50')
plt.bar(x + w/2, pct_df['p99'], width=w, label='p99')
plt.xticks(x, pct_df['method'], rotation=0)
plt.ylabel('E2E delay (ms)')
plt.title(f'Delay percentiles ({PCT_SCENARIO}, task={TASK_DELAY})')
plt.grid(True, axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6) Failure/timeout rate bar chart (timeout default 1s)

FAIL_SCENARIO = 'ge'
sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == FAIL_SCENARIO)].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms'])] 

# Define 'failed_or_timed_out' using the notebook timeout threshold.
sub['failed_or_timed_out'] = (sub['success'] != 1) | (sub['e2e_delay_ms'] > float(TIMEOUT_MS))

rows = []
for m in METHOD_ORDER:
    dm = sub[sub['method'] == m]
    if len(dm) == 0:
        continue
    rate = float(dm['failed_or_timed_out'].mean())
    rows.append({'method': _method_label(m), 'rate': rate})
rate_df = pd.DataFrame(rows)

plt.figure(figsize=(5.8, 3.0))
plt.bar(rate_df['method'], rate_df['rate'])
plt.ylim(0.0, 1.0)
plt.ylabel('Failure/timeout rate')
plt.title(f'Failure/timeout rate (timeout={TIMEOUT_S}s, {FAIL_SCENARIO})')
plt.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# 7) GE: completion ratio by pi_bad bins (default DDL=350ms)

sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == 'ge')].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms'])] 
sub = sub[np.isfinite(sub['pi_bad_pct'])]

# Bins: <3%, 3–6%, 6–10%, >10%
bins = [0.0, 3.0, 6.0, 10.0, 100.0]
labels = ['<3%', '3–6%', '6–10%', '>10%']
sub['pi_bad_bin'] = pd.cut(sub['pi_bad_pct'], bins=bins, labels=labels, include_lowest=True, right=False)

sub['complete'] = (sub['success'] == 1) & (sub['e2e_delay_ms'] > 0) & (sub['e2e_delay_ms'] <= float(GE_DDL_MS))

agg = (
    sub.groupby(['pi_bad_bin', 'method'], as_index=False)
       .agg(complete_ratio=('complete', 'mean'))
)
agg['method_label'] = agg['method'].apply(_method_label)

plt.figure(figsize=(6.6, 3.2))
sns.barplot(data=agg, x='pi_bad_bin', y='complete_ratio', hue='method_label')
plt.ylim(0.0, 1.02)
plt.xlabel('pi_bad bin')
plt.ylabel(f'Completion ratio (DDL={GE_DDL_MS}ms)')
plt.title('GE completion ratio by pi_bad bins')
plt.grid(True, axis='y', alpha=0.25)
plt.legend(title='Method')
plt.tight_layout()
plt.show()

In [ ]:
# 8) IID: E2E delay vs loss rate boxplot

sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == 'iid') & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['e2e_delay_ms']) & (sub['e2e_delay_ms'] > 0)]
sub = sub[np.isfinite(sub['iid_loss_pct'])]

order = sorted(sub['iid_loss_pct'].unique().tolist())
sub['loss_pct'] = pd.Categorical(sub['iid_loss_pct'], categories=order, ordered=True)
sub['method_label'] = sub['method'].apply(_method_label)

plt.figure(figsize=(7.2, 3.4))
sns.boxplot(data=sub, x='loss_pct', y='e2e_delay_ms', hue='method_label')
plt.xlabel('IID loss rate (%)')
plt.ylabel('E2E delay (ms)')
plt.title('IID: E2E delay vs loss rate')
plt.grid(True, axis='y', alpha=0.25)
plt.legend(title='Method')
plt.tight_layout()
plt.show()

In [ ]:
# 9) IID: Overhead vs loss rate boxplot

sub = df_sel[(df_sel['task'] == TASK_DELAY) & (df_sel['scenario'] == 'iid') & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['overhead_ratio']) & (sub['overhead_ratio'] >= 0)]
sub = sub[np.isfinite(sub['iid_loss_pct'])]

order = sorted(sub['iid_loss_pct'].unique().tolist())
sub['loss_pct'] = pd.Categorical(sub['iid_loss_pct'], categories=order, ordered=True)
sub['method_label'] = sub['method'].apply(_method_label)

plt.figure(figsize=(7.2, 3.4))
sns.boxplot(data=sub, x='loss_pct', y='overhead_ratio', hue='method_label')
plt.xlabel('IID loss rate (%)')
plt.ylabel('Overhead ratio')
plt.title('IID: Overhead vs loss rate')
plt.grid(True, axis='y', alpha=0.25)
plt.legend(title='Method')
plt.tight_layout()
plt.show()

In [ ]:
# 10) Goodput vs overhead scatter

GP_SCENARIO = 'ge'  # 'ge' or 'iid'
GP_AGG = 'sender_loss_method_mean'  # 'per_trial' or 'sender_loss_method_mean'

sub = df_sel[(df_sel['task'] == TASK_GOODPUT) & (df_sel['scenario'] == GP_SCENARIO) & (df_sel['success'] == 1)].copy()
sub = sub[np.isfinite(sub['goodput_mbps']) & (sub['goodput_mbps'] > 0)]
sub = sub[np.isfinite(sub['overhead_ratio']) & (sub['overhead_ratio'] >= 0)]

if GP_AGG == 'sender_loss_method_mean':
    grp = sub.groupby(['sender_id', 'loss_mode', 'method'], as_index=False).agg({
        'goodput_mbps': 'mean',
        'overhead_ratio': 'mean',
    })
else:
    grp = sub

grp['method_label'] = grp['method'].apply(_method_label)

plt.figure(figsize=(6.2, 3.4))
sns.scatterplot(data=grp, x='overhead_ratio', y='goodput_mbps', hue='method_label')
plt.xlabel('Overhead ratio')
plt.ylabel('Goodput (Mbps)')
plt.title(f'Goodput vs overhead ({GP_SCENARIO}, task={TASK_GOODPUT}, {GP_AGG})')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()